In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
import pandas as pd

import mlflow
from mlflow.tracking import MlflowClient
import numpy as np
from collections import defaultdict


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

RobertaModel LOAD REPORT from: AIDA-UPM/star
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
model_name = "AIDA-UPM/star"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [ ]:
TRACKING_URI = "https://average-candied-vendor.ngrok-free.dev"
EXPERIMENT   = "star_model_testing"
RUN_NAME     = "star-1nn-authorship"
MLFLOW_PREPROCESS_RUN_ID = "406f39f4f7844e10b58b1b508d6f2bc7"
#MLFLOW_PREPROCESS_RUN_ID = "e48ec02695424bedaab895917a6b19f1"
MLFLOW_DF_ARTIFACT_RUN_ID = "e48ec02695424bedaab895917a6b19f1"
MLFLOW_DF_ARTIFACT_PATH = "data/multigenres_df.csv"

In [ ]:
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT)
mlflow.start_run(run_name=RUN_NAME)

mlflow.log_param('model_name', model_name)
mlflow.log_param('device', device)

client = MlflowClient()

In [ ]:
#local_path = client.download_artifacts(run_id=MLFLOW_DF_ARTIFACT_RUN_ID, path=MLFLOW_DF_ARTIFACT_PATH)
local_path = "balanced_lit.csv"
data = pd.read_csv(local_path)
#data = data[data['source_type'] == 'socials']
data.head()

In [ ]:
def get_embedding(texts: list[str], batch_size=16) -> torch.Tensor:
    """
    Принимает список текстов, возвращает матрицу эмбеддингов [N, D]
    """
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            output = model(**encoded)
            # Mean pooling по токенам (как в статье)
            attention_mask = encoded["attention_mask"]
            token_embeddings = output.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(
                token_embeddings.size()
            ).float()
            embeddings = torch.sum(
                token_embeddings * input_mask_expanded, 1
            ) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

        all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)

In [ ]:
class EmbeddingStore:
    """
    In-memory хранилище эмбеддингов.
    В продакшене методы search/get_matrix заменяются запросами к pgvector.
    """
    def __init__(self):
        self.embeddings: list = []
        self.authors:    list = []
        self.texts:      list = []
        self.sources:    list = []
        self._matrix_cache = None
        self._dirty: bool = True

    def add(self, author: str, embedding: np.ndarray,
            text: str = '', source: str = 'unknown'):
        self.embeddings.append(embedding)
        self.authors.append(author)
        self.texts.append(text)
        self.sources.append(source)
        self._dirty = True

    def get_matrix(self) -> np.ndarray:
        if self._dirty or self._matrix_cache is None:
            self._matrix_cache = np.stack(self.embeddings)
            self._dirty = False
        return self._matrix_cache

    def get_authors(self) -> list:
        return self.authors

    def search(self, query: np.ndarray, k: int = 5) -> list:
        """Косинусный поиск k ближайших соседей."""
        if not self.embeddings:
            return []
        matrix = self.get_matrix()
        sims   = (matrix @ query.reshape(-1, 1)).squeeze()
        top_k  = np.argsort(sims)[::-1][:k]
        return [
            {'author': self.authors[i],
             'similarity': float(sims[i]),
             'text': self.texts[i][:80]}
            for i in top_k
        ]

    def __len__(self):
        return len(self.embeddings)


In [ ]:
def build_star_embedding_store(df: 'pd.DataFrame',
                               batch_size: int = 16,
                               text_col:   str = 'text',
                               author_col: str = 'author',
                               source_col: str = 'source_id') -> EmbeddingStore:
    """
    Батчевый проход через STAR — строим EmbeddingStore для всего корпуса.
    Эмбеддинги L2-нормализованы (dot product = cosine similarity).
    """
    store   = EmbeddingStore()
    texts   = df[text_col].tolist()
    authors = df[author_col].tolist()
    sources = (df[source_col].tolist() if source_col in df.columns
               else ['unknown'] * len(texts))

    print(f'Строим store: {len(texts)} фрагментов, '
          f'{df[author_col].nunique()} авторов...')

    embs = F.normalize(get_embedding(texts, batch_size=batch_size), dim=1).numpy()

    for i, emb in enumerate(embs):
        store.add(author=authors[i], embedding=emb,
                  text=texts[i], source=sources[i])

    print(f'Store готов: {len(store)} эмбеддингов, {len(set(authors))} авторов')
    return store


def build_weighted_centroids(store: EmbeddingStore) -> EmbeddingStore:
    """
    Центроид автора = взвешенное среднее его эмбеддингов.
    Вес = длина текста (прокси информативности фрагмента).
    """
    author_data = defaultdict(list)
    for i, author in enumerate(store.authors):
        weight = min(len(store.texts[i]), 2000) / 2000.0
        author_data[author].append((store.embeddings[i], weight, store.sources[i]))

    centroid_store = EmbeddingStore()
    for author, items in author_data.items():
        embs, weights, sources = zip(*items)
        weights  = np.array(weights)
        weights  = weights / weights.sum()
        centroid = (np.stack(embs) * weights[:, np.newaxis]).sum(axis=0)
        norm     = np.linalg.norm(centroid)
        centroid = centroid / norm if norm > 0 else centroid
        centroid_store.add(
            author=author,
            embedding=centroid,
            text=f'[centroid: {len(items)} фрагментов]',
            source=sources[0]
        )

    print(f'Centroid store: {len(centroid_store)} авторов')
    return centroid_store


In [ ]:
def predict_star(text: str,
                 fragment_store: EmbeddingStore,
                 centroid_store: EmbeddingStore,
                 k: int                 = 5,
                 threshold: float       = 0.3,
                 fragment_weight: float = 0.35,
                 centroid_weight: float = 0.65) -> dict:
    """
    Гибридное предсказание: фрагменты (max cosine) + центроиды, взвешенно.
    threshold: если лучший score < threshold → 'unknown'.
    """
    emb = F.normalize(get_embedding([text]), dim=1).numpy().squeeze()  # (D,)

    # Поиск по фрагментам — берём максимальное сходство на автора
    frag_results = fragment_store.search(emb, k=k * 5)
    frag_scores  = defaultdict(float)
    for r in frag_results:
        frag_scores[r['author']] = max(frag_scores[r['author']], r['similarity'])

    # Поиск по центроидам
    centroid_results = centroid_store.search(emb, k=k)
    centroid_scores  = {r['author']: r['similarity'] for r in centroid_results}

    # Взвешенное объединение
    all_auth     = set(frag_scores) | set(centroid_scores)
    final_scores = {
        a: fragment_weight * frag_scores.get(a, 0.0)
         + centroid_weight * centroid_scores.get(a, 0.0)
        for a in all_auth
    }

    ranked      = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    best_author, best_score = ranked[0]
    predicted   = best_author if best_score >= threshold else 'unknown'

    return {
        'predicted':       predicted,
        'confidence':      round(best_score, 4),
        'top_k':           [(a, round(s, 4)) for a, s in ranked[:k]],
        'unknown':         predicted == 'unknown',
        'centroid_scores': centroid_scores,
        'frag_scores':     dict(frag_scores),
    }


In [ ]:
def split_corpus(df: pd.DataFrame,
                val_size: float = 0.1) -> tuple:
  """
  Разбивка без утечки: фрагменты одного исходного текста
  попадают только в train или только в val.
  Разбиваем по авторам если source_id недоступен.
  """
  from sklearn.model_selection import GroupShuffleSplit

  groups   = df['source_id'] if 'source_id' in df.columns else df['author']
  splitter = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=42)
  train_idx, val_idx = next(splitter.split(df, groups=groups))

  train_df = df.iloc[train_idx].reset_index(drop=True)
  val_df   = df.iloc[val_idx].reset_index(drop=True)

  print(f"Train: {len(train_df)} фрагментов, {train_df['author'].nunique()} авторов")
  print(f"Val:   {len(val_df)} фрагментов, {val_df['author'].nunique()} авторов")
  return train_df, val_df

Датасет: 29773 строк, 103 авторов
Train: 26808 фрагментов, 103 авторов
Val:   2965 фрагментов, 100 авторов


In [ ]:
TRAIN_TEST_SPLIT = 0.2

print("Разбиваем датасет 80/20 без утечки по source_id...")
reference_df, test_eval_df = split_corpus(data, val_size=TRAIN_TEST_SPLIT)

print("Строим fragment store (reference)...")
reference_store = build_star_embedding_store(reference_df)

print("Строим centroid store...")
centroid_store  = build_weighted_centroids(reference_store)


In [ ]:
print(len(reference_df), len(test_eval_df))

2965

## UMAP-визуализация эмбеддингов

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


def get_embeddings_for_viz(store: EmbeddingStore,
                            max_per_author: int = 50):
    author_indices = defaultdict(list)
    for i, author in enumerate(store.authors):
        author_indices[author].append(i)

    selected_indices, selected_authors = [], []
    for author, indices in author_indices.items():
        chosen = indices[:max_per_author]
        selected_indices.extend(chosen)
        selected_authors.extend([author] * len(chosen))

    matrix = np.stack(store.embeddings)[selected_indices]
    return matrix, selected_authors


def plot_star_2d(store: EmbeddingStore,
                 method: str = 'umap',
                 max_per_author: int = 50,
                 figsize: tuple = (14, 10)):
    import umap

    matrix, authors = get_embeddings_for_viz(store, max_per_author)

    n_pca   = min(50, matrix.shape[1], matrix.shape[0] - 1)
    reduced = PCA(n_components=n_pca, random_state=42).fit_transform(matrix)

    if method == 'umap':
        import umap as umap_lib
        coords = umap_lib.UMAP(
            n_components=2, n_neighbors=15, min_dist=0.1,
            metric='cosine', random_state=42
        ).fit_transform(reduced)
    else:
        from sklearn.manifold import TSNE
        coords = TSNE(
            n_components=2, perplexity=min(30, len(matrix) // 5),
            random_state=42, n_iter=1000, metric='cosine'
        ).fit_transform(reduced)

    unique_authors = sorted(set(authors))
    cmap   = plt.cm.get_cmap('tab20' if len(unique_authors) <= 20 else 'hsv',
                             len(unique_authors))
    colors = {a: cmap(i) for i, a in enumerate(unique_authors)}

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor('#0f0f1a')
    fig.patch.set_facecolor('#0f0f1a')

    for author in unique_authors:
        mask = [a == author for a in authors]
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=[colors[author]], label=author,
                   alpha=0.7, s=25, edgecolors='none')

    ax.set_title(f'Эмбеддинги авторов (STAR, {method.upper()})',
                 color='white', fontsize=14, pad=15)
    ax.tick_params(colors='gray')
    for spine in ax.spines.values():
        spine.set_color('#333355')
    ax.legend(
        bbox_to_anchor=(1.02, 1), loc='upper left',
        fontsize=7, ncol=max(1, len(unique_authors) // 25),
        facecolor='#1a1a2e', edgecolor='#333355', labelcolor='white'
    )
    plt.tight_layout()
    fname = f'star_embeddings_{method}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
    mlflow.log_artifact(fname, f'plots/{fname}')
    plt.show()
    print(f'Сохранено и залогировано: {fname}')
    return fig


# Fragment store — полное пространство эмбеддингов
fig_frag = plot_star_2d(reference_store, method='umap')

# Centroid store — по одной точке на автора
fig_cent = plot_star_2d(centroid_store, method='umap')
mlflow.log_artifact('star_embeddings_umap.png', 'plots/star_embeddings_centroids_umap.png')


## Предсказание автора для конкретного текста

In [ ]:
sample_idx         = 0
sample_text        = test_eval_df['text'].iloc[sample_idx]
sample_true_author = test_eval_df['author'].iloc[sample_idx]

result = predict_star(
    text           = sample_text,
    fragment_store = reference_store,
    centroid_store = centroid_store,
)

print(f'Текст (первые 100 символов): {sample_text[:100]}...')
print(f'Истинный автор: {sample_true_author}')
print(f'Предсказанный:  {result["predicted"]} (confidence={result["confidence"]})')
print(f'\nTop-{len(result["top_k"])}:')
for author, score in result['top_k']:
    marker = '\u2713' if author == sample_true_author else ' '
    print(f'  {marker} {author}: {score}')


## Classification Report (1-NN, train/test split)

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)


def evaluate_star_1nn(reference_store: EmbeddingStore,
                      test_df: 'pd.DataFrame',
                      batch_size: int = 16,
                      print_report: bool = True) -> tuple:
    """
    Оценка STAR по 1-NN.
    reference_store — EmbeddingStore из train-корпуса (база известных авторов).
    test_df         — тестовая выборка.
    Возвращает (metrics_dict, classification_report_str).
    """
    print('Вычисляем тестовые эмбеддинги...')
    test_texts   = test_df['text'].tolist()
    true_authors = test_df['author'].tolist()
    test_emb     = F.normalize(
        get_embedding(test_texts, batch_size=batch_size), dim=1
    ).numpy()                                               # (N_test, D)

    store_matrix  = reference_store.get_matrix()           # (N_ref, D)
    store_authors = reference_store.get_authors()

    # 1-NN cosine (эмбеддинги уже L2-нормализованы → dot = cosine)
    sim_matrix   = test_emb @ store_matrix.T               # (N_test, N_ref)
    nearest_idx  = np.argmax(sim_matrix, axis=1)
    pred_authors = [store_authors[i] for i in nearest_idx]

    known_authors = set(store_authors)
    labels        = sorted(set(true_authors) & known_authors)

    mask   = [a in known_authors for a in true_authors]
    y_true = [a for a, m in zip(true_authors, mask) if m]
    y_pred = [a for a, m in zip(pred_authors, mask) if m]

    n_skipped = sum(1 for m in mask if not m)
    if n_skipped:
        print(f'\u26a0 {n_skipped} текстов пропущено: авторов нет в reference store')

    results = {
        'accuracy':           accuracy_score(y_true, y_pred),
        'precision_macro':    precision_score(y_true, y_pred, labels=labels,
                                             average='macro',    zero_division=0),
        'recall_macro':       recall_score(y_true, y_pred, labels=labels,
                                          average='macro',    zero_division=0),
        'f1_macro':           f1_score(y_true, y_pred, labels=labels,
                                      average='macro',    zero_division=0),
        'f1_weighted':        f1_score(y_true, y_pred, labels=labels,
                                      average='weighted', zero_division=0),
        'n_test':             len(y_true),
        'n_authors':          len(labels),
        'y_true':             y_true,
        'y_pred':             y_pred,
        'labels':             labels,
    }

    if print_report:
        print(f"\n{'\u2550'*60}")
        print(f"  STAR 1-NN Classification Report"
              f" ({len(y_true)} текстов, {len(labels)} авторов)")
        print(f"{'\u2550'*60}")
        print(f"  Accuracy:       {results['accuracy']:.4f}")
        print(f"  F1 (macro):     {results['f1_macro']:.4f}")
        print(f"  F1 (weighted):  {results['f1_weighted']:.4f}")
        print(f"{'\u2500'*60}")
        print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    cr_text          = classification_report(y_true, y_pred, labels=labels, zero_division=0)
    results['cr_text'] = cr_text
    return results, cr_text




In [ ]:

star_metrics, star_cr_text = evaluate_star_1nn(reference_store, test_eval_df)

In [ ]:
# ── Логирование метрик и артефактов в MLflow ────────────────────────────────
import json as _json

mlflow.log_param('train_test_split', TRAIN_TEST_SPLIT)
mlflow.log_param('n_authors',        star_metrics['n_authors'])
mlflow.log_param('n_test',           star_metrics['n_test'])

mlflow.log_metric('accuracy',        star_metrics['accuracy'])
mlflow.log_metric('f1_macro',        star_metrics['f1_macro'])
mlflow.log_metric('f1_weighted',     star_metrics['f1_weighted'])
mlflow.log_metric('precision_macro', star_metrics['precision_macro'])
mlflow.log_metric('recall_macro',    star_metrics['recall_macro'])

with open('star_cr.txt', 'w') as f:
    f.write(star_cr_text)
mlflow.log_artifact('star_cr.txt', 'data/star_cr.txt')

_metrics_save = {k: v for k, v in star_metrics.items()
                 if k not in ('y_true', 'y_pred', 'labels', 'cr_text')}
with open('star_metrics.json', 'w') as f:
    _json.dump(_metrics_save, f, indent=4)
mlflow.log_artifact('star_metrics.json', 'data/star_metrics.json')

print('Метрики и артефакты залогированы в MLflow')


## Confusion Matrix

In [ ]:
import seaborn as sns

y_true = star_metrics['y_true']
y_pred = star_metrics['y_pred']
labels = star_metrics['labels']

cm      = confusion_matrix(y_true, y_pred, labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

n       = len(labels)
figsize = max(14, n * 0.45)
fig, ax = plt.subplots(figsize=(figsize, figsize))

sns.heatmap(
    cm_norm, ax=ax,
    xticklabels=labels, yticklabels=labels,
    cmap='Blues', vmin=0, vmax=1,
    linewidths=0.3, linecolor='#e0e0e0',
    annot=(n <= 20), fmt='.2f' if n <= 20 else '',
    annot_kws={'size': 7},
    cbar_kws={'label': 'Доля предсказаний (по строке)'},
)
ax.set_xlabel('Предсказанный автор', fontsize=12, labelpad=10)
ax.set_ylabel('Истинный автор',      fontsize=12, labelpad=10)
ax.set_title(
    f'Матрица ошибок STAR (1-NN, нормализована по строке)\n'
    f'Accuracy={star_metrics["accuracy"]:.3f}  '
    f'F1-macro={star_metrics["f1_macro"]:.3f}  '
    f'n={star_metrics["n_test"]} текстов, {n} авторов',
    fontsize=13, pad=14
)
ax.tick_params(axis='x', rotation=90, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)
plt.tight_layout()
plt.savefig('star_confusion_matrix.png', dpi=150, bbox_inches='tight')
mlflow.log_figure(fig, 'plots/star_confusion_matrix.png')
plt.show()

print('\nТоп-10 самых частых ошибок:')
errors = []
for i, true_a in enumerate(labels):
    for j, pred_a in enumerate(labels):
        if i != j and cm[i, j] > 0:
            errors.append((cm[i, j], cm_norm[i, j], true_a, pred_a))
errors.sort(reverse=True)
print(f'  {"Истинный":<25} {"Предсказан как":<25} {"Кол-во":>7}  {"Доля":>6}')
print('  ' + '-' * 68)
for cnt, share, true_a, pred_a in errors[:10]:
    print(f'  {true_a:<25} {pred_a:<25} {int(cnt):>7}  {share:>6.2%}')


## Разбор предсказаний по выбранным авторам

In [ ]:
import json

AUTHORS_TO_INSPECT = ['Bulgakov', 'Turgenev', 'Роулинг']

y_true = star_metrics['y_true']
y_pred = star_metrics['y_pred']

inspect_data = {'authors': []}

for author in AUTHORS_TO_INSPECT:
    indices = [i for i, a in enumerate(y_true) if a == author]
    if not indices:
        print(f'[{author}] — не найден в тестовой выборке\n')
        continue

    preds = [y_pred[i] for i in indices]
    total = len(preds)
    counts: dict = {}
    for p in preds:
        counts[p] = counts.get(p, 0) + 1
    counts = sorted(counts.items(), key=lambda x: -x[1])

    predictions = [
        {'predicted_label': label, 'percentage': round(cnt / total * 100, 2)}
        for label, cnt in counts
    ]
    inspect_data['authors'].append({'true_author': author, 'predictions': predictions})

    correct = sum(c for p, c in counts if p == author)
    print(f'\u250c\u2500 {author} ({total} фрагментов, accuracy {correct/total:.1%})')
    for pred_label, cnt in counts:
        pct  = cnt / total
        bar  = '\u2588' * int(pct * 30)
        mark = ' \u2713' if pred_label == author else ''
        print(f'\u2502  {pred_label:<28} {bar:<30} {pct:>6.1%}  ({cnt}){mark}')
    print('\u2514' + '\u2500' * 70 + '\n')

with open('star_author_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(inspect_data, f, ensure_ascii=False, indent=2)
mlflow.log_artifact('star_author_predictions.json',
                    'data/star_author_predictions.json')
print('Сохранено и залогировано в MLflow: star_author_predictions.json')


In [ ]:
mlflow.end_run()
